# Question 3 experiment: choosing the submission settings

This notebook is my scratch record for comparing settings on the supplied news data since the first run had pretty bad results lol. The first trial is the original **8-epoch, 128-word, batch-64, learning-rate-0.001** recipe suggested by an [LLM](https://chatgpt.com/share/6ad1a34b-5438-83ec-96a0-d63bff126d1c). Later trials change one setting at a time along the sequence.

The selected recipe is implemented directly in [the main submission notebook](../Question3/submission_model.ipynb): **192 words, batch size 32, learning rate 0.003, and 25 epochs**.


In [ ]:
import csv
import json
import random
import re
from collections import Counter
from pathlib import Path

import torch
from torch import nn

for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent / 'ICAIC_Outputs' / 'Question1').is_dir():
        OUTPUTS_ROOT = parent / 'ICAIC_Outputs'
        break
    if (parent / 'Question1').is_dir():
        OUTPUTS_ROOT = parent
        break

TRAIN_PATH = OUTPUTS_ROOT / 'Question3' / 'train_v2' / 'train_news.csv'
TEST_PATH = TRAIN_PATH.parent / 'test_news_nolabel.csv'
OUTPUT_DIR = OUTPUTS_ROOT / 'Question3-Extension'
OUTPUT_DIR.mkdir(exist_ok=True)
torch.set_num_threads(4)
torch.use_deterministic_algorithms(True)


def tokenize(text, max_length):
    return re.findall("[a-z]+(?:'[a-z]+)?", text.lower())[:max_length]


with TRAIN_PATH.open(newline='', encoding='utf-8-sig') as file:
    rows = list(csv.DictReader(file))
categories = sorted({row['category'] for row in rows})
texts, answer_ids, seen = [], [], set()
for row in rows:
    key = tuple(tokenize(row['text'], 128))
    if key in seen:
        continue
    seen.add(key)
    texts.append(row['text'])
    answer_ids.append(categories.index(row['category']))
labels = torch.tensor(answer_ids, dtype=torch.long)
print('Reading:', TRAIN_PATH)
print('CSV rows:', len(rows), '; distinct first-128-word inputs:', len(texts))
print('Repeated rows removed:', len(rows) - len(texts))
for index, category in enumerate(categories):
    print(category, ':', (labels == index).sum().item())


Reading: /Users/davidfrancisco/Documents/ICAIC_Outputs/Question3/train_v2/train_news.csv
CSV rows: 1000 ; distinct first-128-word inputs: 970
Repeated rows removed: 30
business : 287
entertainment : 199
sport : 281
tech : 203


In [2]:
def split_rows(indices, labels, seed):
    rng = random.Random(seed)
    answers = labels.reshape(-1).tolist()
    learning, checking = [], []
    for label in sorted(set(answers)):
        group = [index for index in indices if answers[index] == label]
        rng.shuffle(group)
        count = round(0.2 * len(group))
        checking.extend(group[:count])
        learning.extend(group[count:])
    rng.shuffle(learning)
    rng.shuffle(checking)
    return torch.tensor(learning), torch.tensor(checking)

development, holdout = split_rows(list(range(len(labels))), labels, 42)
training, validation = split_rows(development.tolist(), labels, 43)
print('Training:', len(training), '; validation:', len(validation), '; holdout:', len(holdout))


Training: 621 ; validation: 155 ; holdout: 194


## 2. Convert each trial's text into numbers

The dictionary maps each known word to an ID wherein ID 0 is padding and ID 1 is unknown. Words need at least two occurrences with a maximum of 12,000 entries in which each trial's vocabulary is built only from its training text.

The model looks up a learned 64-number vector for each word ID, processes the sequence with the LSTM, averages outputs for real words, applies dropout during training, and calculates four topic scores. And then the largest score it returns predicts the category.


In [3]:
def build_vocabulary(token_lists):
    counts = Counter(word for words in token_lists for word in words)
    vocabulary = {'<PAD>': 0, '<UNK>': 1}
    for word, count in sorted(counts.items(), key=lambda item: (-item[1], item[0])):
        if count >= 2 and len(vocabulary) < 12000:
            vocabulary[word] = len(vocabulary)
    return vocabulary


def encode(token_lists, vocabulary, max_length):
    encoded = torch.zeros(len(token_lists), max_length, dtype=torch.long)
    for index, words in enumerate(token_lists):
        ids = [vocabulary.get(word, 1) for word in words[:max_length]] or [1]
        encoded[index, :len(ids)] = torch.tensor(ids)
    return encoded


def prepare(indices, max_length):
    tokens = [tokenize(text, max_length) for text in texts]
    vocabulary = build_vocabulary([tokens[index] for index in indices])
    return encode(tokens, vocabulary, max_length), vocabulary


In [4]:
class MyModel(nn.Module):
    def __init__(self, vocabulary_size, number_of_categories):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, 64, padding_idx=0)
        self.lstm = nn.LSTM(64, 64, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.fc1 = nn.Linear(64, number_of_categories)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.lstm(embedded)
        mask = (x != 0).unsqueeze(-1)
        pooled = (output * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        return self.fc1(self.dropout(pooled))


## 3. Train and calculate the scores

Accuracy counts the fraction correct, while macro F1 calculates F1 for each topic, then averages the four scores equally. Each trial starts from seed 42 and then Adam and cross-entropy update the embedding, LSTM, and linear parameters. The model is checked on validation after each epoch in which the highest macro F1 chooses the stopping epoch.

In [5]:
@torch.no_grad()
def predict_scores(model, features, batch_size=64):
    model.eval()
    return torch.cat([model(features[start:start + batch_size])
                      for start in range(0, len(features), batch_size)])


def classification_metrics(labels, predictions):
    scores = []
    for index in range(len(categories)):
        actual, predicted = labels == index, predictions == index
        tp = (actual & predicted).sum().item()
        fp = (~actual & predicted).sum().item()
        fn = (actual & ~predicted).sum().item()
        denominator = 2 * tp + fp + fn
        scores.append(2 * tp / denominator if denominator else 0.0)
    correct = (labels == predictions).sum().item()
    return {'macro_f1': sum(scores) / len(categories), 'accuracy': correct / len(labels),
            'correct': correct, 'total': len(labels)}


def evaluate(model, features, labels):
    scores = predict_scores(model, features)
    metrics = classification_metrics(labels, scores.argmax(dim=1))
    metrics['cross_entropy'] = nn.CrossEntropyLoss()(scores, labels).item()
    return metrics


def fit(features, labels, vocabulary_size, config, epochs, validation_data=None):
    torch.manual_seed(42)
    model = MyModel(vocabulary_size, len(categories))
    optimizer = torch.optim.Adam(model.parameters(), lr=config['learning_rate'], weight_decay=0.0001)
    generator = torch.Generator().manual_seed(1042)
    best_key, best_epoch, best_metrics = None, epochs, None
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(len(features), generator=generator)
        for start in range(0, len(features), config['batch_size']):
            batch = order[start:start + config['batch_size']]
            optimizer.zero_grad(set_to_none=True)
            loss = nn.CrossEntropyLoss()(model(features[batch]), labels[batch])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        if validation_data is not None:
            metrics = evaluate(model, *validation_data)
            key = (metrics['macro_f1'], -metrics['cross_entropy'])
            if best_key is None or key > best_key:
                best_key, best_epoch, best_metrics = key, epoch, metrics
        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            message = f'Epoch {epoch}/{epochs}'
            if validation_data is not None:
                message += f"; validation macro F1={metrics['macro_f1']:.6f}"
            print(message, flush=True)
    model.eval()
    return model, best_epoch, best_metrics


## 4. Compare five recipes

The trials change one setting at a time along this sequence like making more epochs, higher learning rate, smaller batches, then more words per article. The rows show the best validation epoch within each run. All trials keep the same article IDs and split for cohesion.

In [6]:
trials = [
    {'name': 'Original 8 epochs', 'max_length': 128, 'epochs': 8, 'batch_size': 64, 'learning_rate': 0.001},
    {'name': 'More epochs', 'max_length': 128, 'epochs': 30, 'batch_size': 64, 'learning_rate': 0.001},
    {'name': 'Higher learning rate', 'max_length': 128, 'epochs': 30, 'batch_size': 64, 'learning_rate': 0.003},
    {'name': 'Smaller batches', 'max_length': 128, 'epochs': 30, 'batch_size': 32, 'learning_rate': 0.003},
    {'name': 'Longer articles', 'max_length': 192, 'epochs': 30, 'batch_size': 32, 'learning_rate': 0.003},
]
results = []
for config in trials:
    print('\nTrial:', config['name'], flush=True)
    features, vocabulary = prepare(training.tolist(), config['max_length'])
    _, epoch, metrics = fit(features[training], labels[training], len(vocabulary), config, config['epochs'],
                            validation_data=(features[validation], labels[validation]))
    results.append({**config, 'selected_epoch': epoch, 'vocabulary_size': len(vocabulary), **metrics})

print('\nTrial | words | selected epoch | validation accuracy | macro F1')
for row in results:
    print(f"{row['name']} | {row['max_length']} | {row['selected_epoch']} | {row['accuracy']:.2%} | {row['macro_f1']:.6f}")
with (OUTPUT_DIR / 'validation_results.csv').open('w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=list(results[0]))
    writer.writeheader()
    writer.writerows(results)
selected = max(results, key=lambda row: (row['macro_f1'], -row['cross_entropy']))
print('Selected using validation:', selected)



Trial: Original 8 epochs
Epoch 1/8; validation macro F1=0.290474
Epoch 8/8; validation macro F1=0.478266

Trial: More epochs
Epoch 1/30; validation macro F1=0.290474
Epoch 10/30; validation macro F1=0.564708
Epoch 20/30; validation macro F1=0.734226
Epoch 30/30; validation macro F1=0.744218

Trial: Higher learning rate
Epoch 1/30; validation macro F1=0.136854
Epoch 10/30; validation macro F1=0.799683
Epoch 20/30; validation macro F1=0.788298
Epoch 30/30; validation macro F1=0.796175

Trial: Smaller batches
Epoch 1/30; validation macro F1=0.255332
Epoch 10/30; validation macro F1=0.822676
Epoch 20/30; validation macro F1=0.815934
Epoch 30/30; validation macro F1=0.870444

Trial: Longer articles
Epoch 1/30; validation macro F1=0.112500
Epoch 10/30; validation macro F1=0.782962
Epoch 20/30; validation macro F1=0.890315
Epoch 30/30; validation macro F1=0.850164

Trial | words | selected epoch | validation accuracy | macro F1
Original 8 epochs | 128 | 8 | 60.65% | 0.478266
More epochs | 12

## Save the selected parameters

The comparison selected the recipe with the highest validation macro F1 wherein the small JSON file below records that selection.

In [7]:
selected_parameters = {
    'epochs': selected['selected_epoch'],
    'batch_size': selected['batch_size'],
    'learning_rate': selected['learning_rate'],
    'max_length': selected['max_length'],
    'seed': 42,
    'deduplication_length': 128,
    'validation_macro_f1': selected['macro_f1'],
}
(OUTPUT_DIR / 'selected_parameters.json').write_text(json.dumps(selected_parameters, indent=2) + '\n')
print('Selected parameters:', selected_parameters)


Selected parameters: {'epochs': 25, 'batch_size': 32, 'learning_rate': 0.003, 'max_length': 192, 'seed': 42, 'deduplication_length': 128, 'validation_macro_f1': 0.9117326590869028}


## Conclusion from the comparisons

| Recipe | Word limit | Selected epoch | Validation macro F1 |
| --- | ---: | ---: | ---: |
| Original 8 epochs | 128 | 8 | 0.478266 |
| More epochs | 128 | 25 | 0.768487 |
| Higher learning rate | 128 | 26 | 0.823809 |
| Smaller batches | 128 | 27 | 0.881044 |
| Longer articles | 192 | 25 | **0.911733** |

The longer-article recipe won among these trials so now, I've updated it to become the main submission recipe.